In [ ]:
#Feature 1 - The Chat Parser

import numpy as np

with open('hostel_bois.txt', 'r', encoding='utf-8') as f:
 lines = f.readlines()
print("total lines: ", len(lines))

total lines:  3178


In [ ]:
#Feature 2 - Group Overview

messages = []

system_messages = 0
media_messages = 0
deleted_messages = 0

for line in lines:
  line = line.strip()
  if line == "":
    continue
  if "-" not in line:
    continue
  try:
    timestamp, remaining = line.split(" - ", 1)
    if ":" not in remaining:
      system_messages += 1
      continue
    sender, message = remaining.split(":", 1)
    if sender == "Media omitted":
      media_messages += 1
    if message == "This message  was deleted":
      deleted_messages += 1
    messages.append((timestamp,
                     sender,
                     message))
  except:
    system_messages += 1

In [ ]:
print("=" * 60)
print("Parser Summary")
print("=" * 60)
print("Total Parsed Messages :", len(messages))
print("System Messages :", system_messages)
print("Media Messages :", media_messages)
print("Deleted Messages :", deleted_messages)
print("\nFirst Five Messages\n")

for msg in messages[:5]:
    print(msg)

Parser Summary
Total Parsed Messages : 3174
System Messages : 4
Media Messages : 0
Deleted Messages : 0

First Five Messages

('01/04/24, 01:17', 'Rahul', ' scene fix')
('01/04/24, 01:17', 'Rahul', ' haan')
('01/04/24, 01:18', 'Rahul', ' kya scene')
('01/04/24, 02:13', 'Rahul', ' abhi free hai?')
('01/04/24, 02:13', 'Rahul', ' abey')


In [ ]:
participants = set()
message_count = {}

for msg in messages:
    sender = msg[1]
    participants.add(sender)
    if sender not in message_count:
        message_count[sender] = 0
    message_count[sender] += 1
total_messages = len(messages)
total_participants = len(participants)

if messages:
  first_date = messages[0][0].split(",")[0]
  last_date = messages[-1][0].split(",")[0]
else:
    first_date = "N/A"
    last_date = "N/A"

In [ ]:
#Feature 3 - Most Active Day and Hour

day_count = {}

for msg in messages:
    date = msg[0].split(",")[0]
    if date not in day_count:
        day_count[date] = 0
    day_count[date] += 1
busiest_day = max(day_count, key=day_count.get)

print("="*60)
print("MOST ACTIVE DAY")
print("="*60)
print(f"Busiest Day : {busiest_day}")
print(f"Messages    : {day_count[busiest_day]}")

hour_count = {}
for msg in messages:
    time = msg[0].split(",")[1].strip()
    hour = int(time.split(":")[0])

    if hour not in hour_count:
        hour_count[hour] = 0
    hour_count[hour] += 1
busiest_hour = max(hour_count, key=hour_count.get)

print("="*60)
print("MOST ACTIVE HOUR")
print("="*60)
print(f"Busiest Hour : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00")
print(f"Messages      : {hour_count[busiest_hour]}")

MOST ACTIVE DAY
Busiest Day : 04/05/24
Messages    : 76
MOST ACTIVE HOUR
Busiest Hour : 18:00 - 19:00
Messages      : 248


In [ ]:
#Feature 4 - Activity Heatmap (NumPy)

import numpy as np

# Create participant list
people = sorted(list(participants))

# Assign row index to each participant
person_index = {}
for i, person in enumerate(people):
    person_index[person] = i

# Create Heatmap Matrix
heatmap = np.zeros((len(people), 24), dtype=int)

# Fill the matrix
for msg in messages:
    sender = msg[1]
    time = msg[0].split(",")[1].strip()
    hour = int(time.split(":")[0])
    row = person_index[sender]
    heatmap[row][hour] += 1

# Display Numerical Matrix
print("=" * 80)
print("NUMERICAL ACTIVITY MATRIX")
print("=" * 80)
print("Participants:", people)
print()
print(heatmap)
print("\n")

# Display Heatmap
print("=" * 100)
print("ACTIVITY HEATMAP (Messages by Hour)")
print("=" * 100)
print("         ", end="")

for hour in range(24):
    print(f"{hour:02}", end=" ")
print()

for i, person in enumerate(people):
    print(f"{person:<9}", end="")
    maximum = max(heatmap[i])
    for value in heatmap[i]:
        if maximum == 0:
            symbol = "."
        else:
            ratio = value / maximum
            if ratio == 0:
                symbol = "."
            elif ratio <= 0.25:
                symbol = "░"
            elif ratio <= 0.50:
                symbol = "▒"
            elif ratio <= 0.75:
                symbol = "▓"
            else:
                symbol = "█"
        print(f" {symbol}", end="")
    print()
print("\n")
print("=" * 100)
print("HEATMAP LEGEND")
print("=" * 100)
print(".  = No Activity")
print("░  = Low Activity")
print("▒  = Moderate Activity")
print("▓  = High Activity")
print("█  = Very High Activity")

NUMERICAL ACTIVITY MATRIX
Participants: ['Aman', 'Karan', 'Neha', 'Priya', 'Rahul', 'Vikas']

[[ 54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7
   16   8  13  11   0  56]
 [  0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27
   25  32  23  14   9   8]
 [  0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47
   62  50  45  27  28  30]
 [  0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40
   38  60  43  32  18   9]
 [  3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49
  105  76  41  92  60  54]
 [  0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3
    2   2   1   1   1   2]]


ACTIVITY HEATMAP (Messages by Hour)
         00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aman      ▓ █ ▓ ▓ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . ▓
Karan     . . . . . . . ░ ▒ ▒ ▓ ▒ █ ▓ █ ▓ ▓ ▓ ▓ █ ▓ ▒ ░ ░
Neha      . . . . . ▒ ░ ░ ▓ █ █ ▒ ▓ ▓ ▒ ░ ▓ █ █ █ ▓ ▒ ▒ ▒
Priya

In [ ]:
#Feature 5 - Top Words

# Stop words
stop_words = {
    "i","is","the","a","an","and","or","to","of","in","on","for",
    "at","it","this","that","are","am","was","were","be","been",
    "you","your","me","my","we","our","they","their","he","she",
    "his","her","them","with","as","if","but","so","do","did",
    "does","have","has","had","will","would","can","could","just"
}

# Characters to remove
punctuation = ".,!?;:'\"()[]{}<>-_/@#$%^&*+=`~|\\"
word_count = {}

# Count words
for msg in messages:
    text = msg[2]

    # Skip media and deleted messages
    if text == "<Media omitted>" or text == "This message was deleted":
        continue
    text = text.lower()

    # Remove punctuation
    for symbol in punctuation:
        text = text.replace(symbol, "")
    words = text.split()

    for word in words:
        if word in stop_words:
            continue
        if word == "":
            continue
        if word not in word_count:
            word_count[word] = 0
        word_count[word] += 1

# Sort words
sorted_words = sorted(
    word_count.items(),
    key=lambda x: x[1],
    reverse=True
)

# Display Top 10
print("=" * 70)
print("TOP 10 MOST USED WORDS")
print("=" * 70)

for word, count in sorted_words[:10]:

    # Scale the bar to a maximum length of 30
    bar = "█" * max(1, count // 10)
    print(f"{word:<15} {bar:<30} {count}")
print("=" * 70)

# Total unique words
print(f"Total Unique Words : {len(word_count)}")

TOP 10 MOST USED WORDS
how             ████████████████████████████████ 321
guys            ███████████████████████████████ 318
about           ███████████████████████████    274
hai             ██████████████████████████     268
today           █████████████████████████      257
which           ████████████████████           202
everyone        ██████████████████             187
telling         █████████████████              179
from            █████████████████              174
up              █████████████████              172
Total Unique Words : 549


In [ ]:
#Feature 6 - Response Speed & Silent Streaks

from datetime import datetime, timedelta

# Average Response Time

response_times = {}
for i in range(1, len(messages)):
    current = messages[i]
    previous = messages[i - 1]

    if current[1] != previous[1]:
        current_time = datetime.strptime(
            current[0],
            "%d/%m/%y, %H:%M"
        )

        previous_time = datetime.strptime(
            previous[0],
            "%d/%m/%y, %H:%M"
        )

        gap = (current_time - previous_time).total_seconds() / 60
        sender = current[1]
        if sender not in response_times:
            response_times[sender] = []
        response_times[sender].append(gap)

print("=" * 70)
print("AVERAGE RESPONSE TIME")
print("=" * 70)

average_response = {}
# Initialize fastest and slowest to avoid NameError if average_response is empty
fastest = "N/A"
slowest = "N/A"

for sender in participants:
    if sender in response_times and len(response_times[sender]) > 0:
        avg = sum(response_times[sender]) / len(response_times[sender])
        average_response[sender] = avg
        if avg < 60:
            print(f"{sender:<10} : {avg:.2f} minutes")
        else:
            print(f"{sender:<10} : {avg/60:.2f} hours")

print()

# -----------------------------
# Longest Silent Streak
# -----------------------------

print("=" * 70)
print("LONGEST SILENT STREAK")
print("=" * 70)

# Find complete date range
start_date = datetime.strptime(
    messages[0][0].split(",")[0],
    "%d/%m/%y"
)
end_date = datetime.strptime(
    messages[-1][0].split(",")[0],
    "%d/%m/%y"
)
all_dates = []
current = start_date
while current <= end_date:
    all_dates.append(current.strftime("%d/%m/%y"))
    current += timedelta(days=1)

# Check streak for every participant

for sender in sorted(participants):
    active_days = set()
    for msg in messages:
        if msg[1] == sender:
            active_days.add(msg[0].split(",")[0])
    longest = 0
    streak = 0
    for day in all_dates:
        if day not in active_days:
            streak += 1
            if streak > longest:
                longest = streak
        else:
            streak = 0
    print(f"{sender:<10} : {longest} days")
print()

# Fastest & Slowest Replier

if average_response:
    fastest = min(average_response, key=average_response.get)
    slowest = max(average_response, key=average_response.get)
    print("=" * 70)
    print("RESPONSE SUMMARY")
    print("=" * 70)
    if average_response[fastest] < 60:
        print(f"Fastest Replier : {fastest} ({average_response[fastest]:.2f} minutes)")
    else:
        print(f"Fastest Replier : {fastest} ({average_response[fastest]/60:.2f} hours)")
    if average_response[slowest] < 60:
        print(f"Slowest Replier : {slowest} ({average_response[slowest]:.2f} minutes)")
    else:
        print(f"Slowest Replier : {slowest} ({average_response[slowest]/60:.2f} hours)")

AVERAGE RESPONSE TIME
Rahul      : 34.95 minutes
Karan      : 36.62 minutes
Neha       : 39.45 minutes
Priya      : 41.99 minutes
Aman       : 55.36 minutes
Vikas      : 46.30 minutes

LONGEST SILENT STREAK
Aman       : 0 days
Karan      : 0 days
Neha       : 0 days
Priya      : 0 days
Rahul      : 0 days
Vikas      : 11 days

RESPONSE SUMMARY
Fastest Replier : Rahul (34.95 minutes)
Slowest Replier : Aman (55.36 minutes)


In [ ]:
#Feature 7 - Personality Archetype Detection

print("=" * 90)
print("PERSONALITY ARCHETYPES")
print("=" * 90)

caring_words = [
    "okay","safe","eat","sleep","please",
    "care","water","drink","reminder",
    "dont","don't","take"
]
funny_words = [
    "lol","lmao","haha","rofl","lmfao","hehe"
]

personality = {} # Initialize personality dictionary outside the loop

for person in sorted(participants):
    total = 0
    spam = 0
    caring = 0
    night = 0
    story = 0
    drama = 0
    comedy = 0
    questions = 0
    active_days = set()
    burst = 0
    max_burst = 0
    for i, msg in enumerate(messages):
        if msg[1] != person:
            continue
        total += 1
        text = msg[2]
        active_days.add(msg[0].split(",")[0])

        # Night Owl

        hour = int(
            msg[0].split(",")[1].split(":")[0]
        )
        if hour >= 23 or hour <= 4:
            night += 1

        # Story Teller

        if len(text.split()) > 30:
            story += 1

        # Drama Queen

        if (
            len(text) > 3 and text.isupper()
        ) or text.count("!") >= 2:
            drama += 1

        # Group Mom

        lower = text.lower()
        for word in caring_words:
            if word in lower:
                caring += 1

        # Comedian

        for word in funny_words:
            if word in lower:
                comedy += 1

        # Question Master

        if text.strip().endswith("?"):
            questions += 1

        # Spammer

        if i == 0:
            burst = 1
        else:
            if messages[i-1][1] == person:
                burst += 1
            else:
                burst = 1
        if burst > max_burst:
            max_burst = burst

    # Ghost Score

    silent_days = 60 - len(active_days)
    scores = {
        "THE SPAMMER": max_burst,
        "THE GROUP MOM": caring,
        "THE NIGHT OWL": night,
        "THE STORY TELLER": story,
        "THE DRAMA QUEEN": drama,
        "THE GHOST": silent_days,
        "THE COMEDIAN": comedy,
        "THE QUESTION MASTER": questions
    }
    archetype = max(scores, key=scores.get)
    personality[person] = archetype # Store the archetype for each person
    print(f"{person:<10} ➜ {archetype}")

PERSONALITY ARCHETYPES
Aman       ➜ THE NIGHT OWL
Karan      ➜ THE STORY TELLER
Neha       ➜ THE DRAMA QUEEN
Priya      ➜ THE GROUP MOM
Rahul      ➜ THE NIGHT OWL
Vikas      ➜ THE GHOST


In [ ]:
# Feature 8 : THE FINAL REPORT

print("=" * 80)
print(" " * 25 + "GROUPDNA FINAL REPORT")
print("=" * 80)

# GROUP OVERVIEW

print("\nGROUP OVERVIEW")
print("-" * 80)
print(f"Period               : {first_date} to {last_date}")
print(f"Total Messages       : {total_messages}")
print(f"Participants         : {total_participants}")
print("\nMESSAGES PER PERSON")

sorted_people = sorted(
    message_count.items(),
    key=lambda x: x[1],
    reverse=True
)
for person, count in sorted_people:
    percentage = (count / total_messages) * 100
    print(f"{person:<12}: {count:>5} ({percentage:.2f}%)")

# GROUP ACTIVITY

print("\n" + "=" * 80)
print("GROUP ACTIVITY")
print("=" * 80)
print(f"Busiest Day          : {busiest_day}")
print(f"Messages             : {day_count[busiest_day]}")
print()
print(f"Busiest Hour         : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00")
print(f"Messages             : {hour_count[busiest_hour]}")

# TOP WORDS

print("\n" + "=" * 80)
print("TOP 10 WORDS")
print("=" * 80)

for word, count in sorted_words[:10]:
    bar = "█" * max(1, count // 10)
    print(f"{word:<15}{bar:<30}{count}")

# RESPONSE SUMMARY

print("\n" + "=" * 80)
print("RESPONSE SUMMARY")
print("=" * 80)
print(f"Fastest Replier      : {fastest}")
print(f"Slowest Replier      : {slowest}")

# PERSONALITY ARCHETYPES

print("\n" + "=" * 80)
print("PERSONALITY ARCHETYPES")
print("=" * 80)

for person in sorted(personality):
    print(f"{person:<12}→ {personality[person]}")

# END

print("\n" + "=" * 80)
print("Thank you for using GroupDNA!")
print("=" * 80)

                         GROUPDNA FINAL REPORT

GROUP OVERVIEW
--------------------------------------------------------------------------------
Period               : 01/04/24 to 30/05/24
Total Messages       : 3174
Participants         : 6

MESSAGES PER PERSON
Rahul       :   953 (30.03%)
Priya       :   718 (22.62%)
Neha        :   635 (20.01%)
Aman        :   490 (15.44%)
Karan       :   354 (11.15%)
Vikas       :    24 (0.76%)

GROUP ACTIVITY
Busiest Day          : 04/05/24
Messages             : 76

Busiest Hour         : 18:00 - 19:00
Messages             : 248

TOP 10 WORDS
how            ████████████████████████████████321
guys           ███████████████████████████████318
about          ███████████████████████████   274
hai            ██████████████████████████    268
today          █████████████████████████     257
which          ████████████████████          202
everyone       ██████████████████            187
telling        █████████████████             179
from           ██